In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
from lib.Forecast.SimpleTransformerForecast import SimpleTransformerForecast
from lib.Forecast.PatchTransformerForecast import PatchTransformerForecast, PatchTransformerForecast2
from lib.Forecast.PatchTransformerExoForecast import PatchTransformerExoForecast
from lib.Dataloaders.VirtualAgentDataset import VirtualAgentDataset
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import numpy as np
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader
import os
import time
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Function to train the model
def train_loop(model,
               train,
               val,
               optimizer,
               scheduler=None,
               patience=5,
               epochs=100,
               lossf=F.mse_loss):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    """_Training loop for the model_
    

    Args:
        model: model to train
        optimizer: pytorch optimizer, for example torch.optim.Adam
        train: training data
        val: validation data
        epochs: number of epochs

    Returns:
        _type_: training history, a dictionary with the training and validation loss for each epoch
    """

    def epoch_loss(dataset):
        data_loss = 0.0
        for i, (x_endo, x_exo, x_cond, y_target) in enumerate(dataset):
            # Spostiamo tutto su GPU
            x_endo = x_endo.to(device)
            x_exo = x_exo.to(device)
            x_cond = x_cond.to(device)
            y = y_target.to(device)

            # Passiamo i 3 tensori al modello
            outputs = model(x_endo, exo=x_exo, cond=x_cond).squeeze(-1) # squeeze(-1) rimuove solo l'ultima dim = 1
            loss = lossf(y.squeeze(1), outputs) # Schiaccia via l'1 in mezzo, diventando (Batch, 10)
            data_loss += loss.item()
        return data_loss / (i + 1)

    def early_stopping(val_loss, patience=5):
        if len(val_loss) > patience:
            if val_loss[-1] > np.mean(val_loss[-(patience + 1):-1]):
                return True

    hist_loss = {'train': [], 'val': []}
    pbar = tqdm(range(epochs))
    for epoch in pbar:  # loop for all the epochs
        # print(f"Epoch {epoch + 1}/{epochs}")
        model.train()
        for i, (x_endo, x_exo, x_cond, y_target) in enumerate(train):
            x_endo = x_endo.to(device)
            x_exo = x_exo.to(device)
            x_cond = x_cond.to(device)
            y = y_target.to(device)

            optimizer.zero_grad()

            # Reset the gradients
            optimizer.zero_grad()

            # Apply the data to the model
            outputs = model(x_endo, exo=x_exo, cond=x_cond).squeeze(-1)
            # Calculate the loss
            loss = lossf(y.squeeze(1), outputs) # Schiaccia via l'1 in mezzo, diventando (Batch, 10)

            # Make the backward pass
            loss.backward()
            optimizer.step()

        if scheduler is not None:
            scheduler.step()

        # Calculate the loss in the training and validation sets
        model.eval()
        with torch.no_grad():
            hist_loss['train'].append(
                epoch_loss(train))
            hist_loss['val'].append(
                epoch_loss(val))

        # Show the loss in the training and validation sets
        pbar.set_postfix({
            'train': hist_loss['train'][-1],
            'val': hist_loss['val'][-1],
            'lr': optimizer.param_groups[0]['lr']
        })

        # If the loss in the validation set does not decrease, stop the training
        if early_stopping(hist_loss['val'], patience):
            print("\nEarly stopping reached!")
            break

    return hist_loss

In [ ]:
# --- NUOVA FUNZIONE DI PLOT ---
# def plot_prediction(model, data, channel=0):
#     # data è la tupla fornita dal VirtualAgentDataset.dataset[idx]
#     x_endo, x_exo, x_cond, y_target = data
#     device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
#     # Aggiungiamo la dimensione del Batch (unsqueeze(0))
#     inputs_endo = x_endo.unsqueeze(0).to(device)
#     inputs_exo = x_exo.unsqueeze(0).to(device)
#     inputs_cond = x_cond.unsqueeze(0).to(device)
    
#     model.eval()
#     with torch.no_grad():
#         pred = model(inputs_endo, exo=inputs_exo, cond=inputs_cond).squeeze().cpu().numpy()

#     # Prepariamo i dati per il plot
#     X_plot = x_endo.cpu().numpy()
#     Y_plot = y_target.squeeze().cpu().numpy()

#     plt.figure(figsize=(12, 6))
#     ax = plt.subplot(1, 2, 1)
#     # Disegniamo la storia passata (solo il canale 0, che è il Self)
#     plt.plot(X_plot[channel, :], label='Lookback (Norma Distanza 3D)', color='blue')
#     plt.title('Dato Storico (Passato)')
#     plt.legend()
    
#     ax = plt.subplot(1, 2, 2)
#     plt.plot(Y_plot, label='True Future', color='green', linestyle='dashed')
#     plt.plot(pred, label='Predicted Future', color='red')
#     plt.title('Predizione (Futuro)')
#     plt.legend()
#     plt.show()

# # --- NUOVA FUNZIONE DI PLOT ---
# def plot_prediction(model, data, titolo_extra=""):
#     """
#     data: tupla (x_endo, x_exo, x_cond, y_target) fornita dal Dataloader
#     """
#     x_endo, x_exo, x_cond, y_target = data
#     device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
#     # 1. Aggiungiamo la dimensione del Batch (unsqueeze(0)) per dare i dati al modello
#     inputs_endo = x_endo.unsqueeze(0).to(device)
#     inputs_exo = x_exo.unsqueeze(0).to(device)
#     inputs_cond = x_cond.unsqueeze(0).to(device)
    
#     # 2. Predizione
#     model.eval()
#     with torch.no_grad():
#         pred = model(inputs_endo, exo=inputs_exo, cond=inputs_cond).squeeze().cpu().numpy()

#     # 3. Prepariamo i dati per il plot (rimuoviamo i tensori)
#     X_plot = x_endo.cpu().numpy()  # Forma: (8 canali, 100 frame)
#     Y_plot = y_target.squeeze().cpu().numpy() # Forma: (25 frame)

#     plt.figure(figsize=(15, 6))
    
#     # ==========================================
#     # GRAFICO 1: IL PASSATO (Tutti i giocatori)
#     # ==========================================
#     ax1 = plt.subplot(1, 2, 1)
    
#     # Plottiamo prima gli "Altri" (Canali da 1 a 7) per metterli in background
#     altri_plottati = False
#     for i in range(1, X_plot.shape[0]):
#         # Se un canale è tutto a 0.0, significa che il sedile era vuoto (nessun giocatore)
#         if np.all(X_plot[i, :] == 0.0):
#             continue 
        
#         # Disegniamo i compagni di squadra in grigio chiaro
#         plt.plot(X_plot[i, :], color='gray', alpha=0.4, linewidth=1.5)
#         altri_plottati = True
        
#     # Aggiungiamo una label fittizia per la legenda se abbiamo trovato almeno un compagno
#     if altri_plottati:
#         plt.plot([], [], color='gray', alpha=0.4, linewidth=1.5, label='Altri (Compagni di barca)')

#     # Plottiamo il "Self" (Canale 0) in primo piano, con un colore forte
#     plt.plot(X_plot[0, :], color='blue', linewidth=3, label='Self (Giocatore Focus)')
    
#     plt.title(f'Passato: Dinamica di Gruppo {titolo_extra}', fontsize=14)
#     plt.xlabel('Frame (Lookback)', fontsize=12)
#     plt.ylabel('Estensione Braccio (0.0 - 1.0)', fontsize=12)
#     plt.legend(loc='upper right')
#     plt.grid(True, alpha=0.3)
    
#     # ==========================================
#     # GRAFICO 2: IL FUTURO (Predizione sul Self)
#     # ==========================================
#     ax2 = plt.subplot(1, 2, 2)
    
#     # Qui confrontiamo solo il Self Reale vs Self Predetto
#     plt.plot(Y_plot, label='Futuro Reale (Ground Truth)', color='green', linewidth=2.5, linestyle='dashed')
#     plt.plot(pred, label='Predizione Rete', color='red', linewidth=2.5)
    
#     plt.title('Futuro: Orizzonte Predittivo sul Self', fontsize=14)
#     plt.xlabel('Frame Futuri (Horizon)', fontsize=12)
#     plt.legend(loc='upper right')
#     plt.grid(True, alpha=0.3)
    
#     # Assicuriamoci che l'asse Y del futuro abbia la stessa scala del passato per coerenza visiva
#     plt.ylim(ax1.get_ylim()) 

#     plt.tight_layout()
#     plt.show()


# --- 
import matplotlib.pyplot as plt
import numpy as np
import torch

def plot_prediction(model, data, titolo_extra=""):
    """
    data: tupla (x_endo, x_exo, x_cond, y_target) fornita dal Dataloader
    """
    x_endo, x_exo, x_cond, y_target = data
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # 1. Aggiungiamo la dimensione del Batch per la rete
    inputs_endo = x_endo.unsqueeze(0).to(device)
    inputs_exo = x_exo.unsqueeze(0).to(device)
    inputs_cond = x_cond.unsqueeze(0).to(device)
    
    # 2. Predizione
    model.eval()
    with torch.no_grad():
        pred = model(inputs_endo, exo=inputs_exo, cond=inputs_cond).squeeze().cpu().numpy()

    # 3. Prepariamo i dati
    X_plot = x_endo.cpu().numpy()  # (8 canali, lookback)
    Y_plot = y_target.squeeze().cpu().numpy() # (horizon)

    plt.figure(figsize=(16, 6))
    
    # ==========================================
    # GRAFICO 1: IL PASSATO (Dinamica di gruppo)
    # ==========================================
    ax1 = plt.subplot(1, 2, 1)
    
    missing_players = 0
    altri_attivi = False
    
    # Plottiamo i compagni di squadra (Canali da 1 a 7)
    for i in range(1, X_plot.shape[0]):
        # Se la media del canale è < 0, significa che è la linea piatta a -1.0 (posto vuoto)
        if np.mean(X_plot[i, :]) < 0.0:
            missing_players += 1
            # Linea tratteggiata rossa sottile per i posti vuoti
            plt.plot(X_plot[i, :], color='salmon', alpha=0.5, linestyle=':', linewidth=1.5)
        else:
            # Compagni reali che stanno remando in grigio
            plt.plot(X_plot[i, :], color='gray', alpha=0.4, linewidth=1.5)
            altri_attivi = True
            
    # Plottiamo il "Self" (Canale 0) in blu scuro
    plt.plot(X_plot[0, :], color='blue', linewidth=3, label='Self (Target)')
    
    # Aggiungiamo il badge con il numero dei giocatori mancanti
    testo_badge = f"Posti Vuoti: {missing_players} su 7"
    colore_badge = 'red' if missing_players > 0 else 'green'
    ax1.text(0.02, 0.95, testo_badge, transform=ax1.transAxes, 
             fontsize=12, fontweight='bold', color=colore_badge,
             bbox=dict(facecolor='white', alpha=0.8, edgecolor=colore_badge, boxstyle='round,pad=0.5'))

    plt.title(f'Passato: Dinamica di Gruppo {titolo_extra}', fontsize=14)
    plt.xlabel('Frame (Lookback)', fontsize=12)
    plt.ylabel('Estensione Braccio [0.0 - 1.0]', fontsize=12)
    
    # Custom Legend
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    legend_elements = [Line2D([0], [0], color='blue', lw=3, label='Self (Target)')]
    if altri_attivi:
        legend_elements.append(Line2D([0], [0], color='gray', lw=1.5, alpha=0.4, label='Compagni attivi'))
    if missing_players > 0:
        legend_elements.append(Line2D([0], [0], color='salmon', lw=1.5, linestyle=':', label='Posti vuoti (-1.0)'))
        
    ax1.legend(handles=legend_elements, loc='upper right')
    ax1.grid(True, alpha=0.3)
    
    # ==========================================
    # GRAFICO 2: IL FUTURO (Predizione)
    # ==========================================
    ax2 = plt.subplot(1, 2, 2)
    
    plt.plot(Y_plot, label='Futuro Reale (Ground Truth)', color='green', linewidth=2.5, linestyle='dashed')
    plt.plot(pred, label='Predizione Rete', color='red', linewidth=2.5)
    
    plt.title('Futuro: Orizzonte Predittivo sul Self', fontsize=14)
    plt.xlabel('Frame Futuri (Horizon)', fontsize=12)
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.ylim(ax1.get_ylim()) # Mantiene la stessa scala Y del grafico sinistro

    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import ipywidgets as widgets
from ipywidgets import interact, Layout

def interactive_rolling_dashboard(model, dataset, stride=10, lookback=100, horizon=25):
    """
    Crea una dashboard interattiva in Jupyter Notebook per esplorare le predizioni.
    Usa degli slider per navigare nel tempo e confrontare il comportamento del target
    con il resto della barca e il metronomo.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.eval()
    
    # Calcoliamo il limite massimo dello slider in base alla grandezza del dataset
    max_idx = len(dataset) - 50 # Lasciamo un po' di margine in fondo
    
    def update_plot(start_idx, num_windows):
        # ==========================================
        # 1. RACCOLTA DATI (Reale e Contesto)
        # ==========================================
        continuous_true = []
        continuous_teammates = []
        continuous_metronome = []
        
        # Preleviamo la prima finestra intera (il Lookback iniziale)
        first_x_endo, first_x_exo, _, _ = dataset[start_idx]
        
        continuous_true.extend(first_x_endo[0, :].numpy()) # Self
        
        # Calcoliamo la media dei compagni (ignorando i posti vuoti a -1.0)
        teammates = first_x_endo[1:, :].numpy()
        teammates_mask = teammates > -0.5
        # Media sicura per evitare divisioni per zero
        team_mean = np.zeros(lookback)
        for t in range(lookback):
            valid_vals = teammates[:, t][teammates_mask[:, t]]
            if len(valid_vals) > 0:
                team_mean[t] = np.mean(valid_vals)
            else:
                team_mean[t] = -1.0 # Se non c'è nessuno
        continuous_teammates.extend(team_mean)
        
        # Canale 3 dell'EXO è il Metronomo (come visto in debug_dashboard)
        continuous_metronome.extend(first_x_exo[3, :].numpy())

        # Estendiamo per le finestre successive (solo la parte nuova data dallo stride)
        for i in range(num_windows):
            idx = start_idx + i
            if idx >= len(dataset): break
            _, x_exo_step, _, y_target = dataset[idx]
            
            nuovi_frame_reali = y_target.squeeze().numpy()[:stride]
            continuous_true.extend(nuovi_frame_reali)
            
            # Per i compagni e il metronomo, estraiamo l'ultimo 'stride' di dati dal lookback successivo
            # per mantenere la linea temporale sincronizzata
            if idx + 1 < len(dataset):
                next_endo, next_exo, _, _ = dataset[idx + 1]
                
                # Compagni (media)
                next_team = next_endo[1:, -stride:].numpy()
                next_mask = next_team > -0.5
                step_team_mean = np.zeros(stride)
                for t in range(stride):
                    val_v = next_team[:, t][next_mask[:, t]]
                    step_team_mean[t] = np.mean(val_v) if len(val_v) > 0 else -1.0
                continuous_teammates.extend(step_team_mean)
                
                # Metronomo
                continuous_metronome.extend(next_exo[3, -stride:].numpy())
            else:
                # Se siamo alla fine, riempiamo con zeri per allineamento
                continuous_teammates.extend(np.zeros(stride))
                continuous_metronome.extend(np.zeros(stride))

        # ==========================================
        # 2. CREAZIONE DELLA FIGURA A DUE PANNELLI
        # ==========================================
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 10), gridspec_kw={'height_ratios': [2, 1]})
        fig.suptitle(f"Analisi Spazio-Temporale | Indice Iniziale: {start_idx} | Orizzonte: {num_windows} finestre", fontsize=16, fontweight='bold')
        
        # --- PANNELLO SUPERIORE: PREDITTIVO ---
        ax1.plot(continuous_true, color='lightgreen', linewidth=4, label='Traiettoria Reale (Ground Truth)')
        
        with torch.no_grad():
            for i in range(num_windows):
                idx = start_idx + i
                if idx >= len(dataset): break
                
                x_endo, x_exo, x_cond, _ = dataset[idx]
                
                inputs_e = x_endo.unsqueeze(0).to(device)
                inputs_ex = x_exo.unsqueeze(0).to(device)
                inputs_c = x_cond.unsqueeze(0).to(device)
                
                pred = model(inputs_e, exo=inputs_ex, cond=inputs_c).squeeze().cpu().numpy()
                
                start_plot_x = lookback + (i * stride)
                x_axis_pred = range(start_plot_x, start_plot_x + horizon)
                
                label = 'Predizione Rete (Horizon)' if i == 0 else ""
                ax1.plot(x_axis_pred, pred, color='red', linewidth=2, alpha=0.8, label=label)
                ax1.plot(start_plot_x, pred[0], marker='o', color='darkred', markersize=5)

        ax1.axvline(x=lookback, color='gray', linestyle='--', linewidth=2, label='Inizio Predizioni Future')
        ax1.set_ylabel("Estensione Braccio (Self)", fontsize=12)
        ax1.set_title("Cinematica del Giocatore (Reale vs Predetto)", fontsize=14)
        ax1.legend(loc='upper right')
        ax1.grid(True, alpha=0.3)
        ax1.set_xlim(0, lookback + (num_windows * stride) + horizon)
        ax1.set_ylim(0, 1.05)

        # --- PANNELLO INFERIORE: CONTESTO ---
        ax2.plot(continuous_teammates, color='gray', linewidth=2, alpha=0.7, label='Media Movimento Compagni')
        ax2.plot(continuous_metronome, color='purple', linestyle=':', linewidth=2, label='Ritmo Metronomo (Hz/2)')
        
        ax2.axvline(x=lookback, color='gray', linestyle='--', linewidth=2)
        ax2.set_xlabel("Frame Temporali Continui", fontsize=12)
        ax2.set_ylabel("Intensità", fontsize=12)
        ax2.set_title("Contesto di Bordo (Team & Ambiente)", fontsize=14)
        ax2.legend(loc='upper right')
        ax2.grid(True, alpha=0.3)
        ax2.set_xlim(0, lookback + (num_windows * stride) + horizon)
        ax2.set_ylim(0, 1.05)

        plt.tight_layout()
        plt.show()

    # ==========================================
    # 3. ATTIVAZIONE SLIDER INTERATTIVI
    # ==========================================
    # Crea un widget per il punto di partenza
    slider_start = widgets.IntSlider(
        value=0, min=0, max=max_idx, step=1, 
        description='Inizio (Idx):', 
        layout=Layout(width='800px'), continuous_update=False
    )
    
    # Crea un widget per la lunghezza del futuro che vuoi guardare
    slider_windows = widgets.IntSlider(
        value=15, min=1, max=50, step=1, 
        description='Finestre Future:', 
        layout=Layout(width='500px'), continuous_update=False
    )
    
    interact(update_plot, start_idx=slider_start, num_windows=slider_windows)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def plot_all_players_full_session(model, dataset, stride=10, lookback=100, horizon=25, max_players=None):
    """
    Scansiona interamente il dataset e genera un plot separato per l'intera 
    sessione di gioco di ogni singolo giocatore rilevato, evidenziando gli ancoraggi.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.eval()
    
    current_idx = 0
    player_count = 0
    
    print(f"Inizio scansione del dataset (Totale campioni: {len(dataset)})...")
    
    while current_idx < len(dataset):
        # Se abbiamo raggiunto il limite di giocatori da plottare, ci fermiamo
        if max_players is not None and player_count >= max_players:
            print(f"Raggiunto il limite massimo di {max_players} giocatori richiesti.")
            break
            
        print(f"\n--- Estrazione Giocatore {player_count + 1} (Inizio a indice {current_idx}) ---")
        
        # 1. IDENTIFICAZIONE DELLA SESSIONE DEL GIOCATORE
        start_idx = current_idx
        _, _, first_x_cond, _ = dataset[start_idx]
        
        # Troviamo dove finisce questo giocatore
        end_idx = start_idx
        while end_idx < len(dataset):
            _, _, curr_x_cond, _ = dataset[end_idx]
            # Se le condizioni (Lato/Spawn) cambiano, è un giocatore diverso
            if not torch.allclose(curr_x_cond, first_x_cond):
                break
            end_idx += 1
            
        num_windows = end_idx - start_idx
        print(f"Trovate {num_windows} finestre valide per questo giocatore.")
        
        # 2. RACCOLTA DEI DATI (Reale e Predetto)
        continuous_true = []
        
        # Preleviamo il lookback iniziale
        first_x_endo, _, _, _ = dataset[start_idx]
        continuous_true.extend(first_x_endo[0, :].numpy())
        
        # Estraiamo tutta la verità (Ground Truth)
        for i in range(num_windows):
            _, _, _, y_target = dataset[start_idx + i]
            nuovi_frame_reali = y_target.squeeze().numpy()[:stride]
            continuous_true.extend(nuovi_frame_reali)
            
        # 3. PLOT DELLA SESSIONE COMPLETA
        plt.figure(figsize=(18, 6))
        plt.title(f"Performance Modello - Giocatore {player_count + 1} (Intera Sessione)", fontsize=16, fontweight='bold')
        
        # Aggiungiamo un'area ombreggiata per il "Warm-up" (Lookback iniziale)
        plt.axvspan(0, lookback, color='lightgray', alpha=0.3, label='Fase di Inizializzazione (Lookback 0)')
        
        # Disegniamo la realtà (Ground Truth)
        plt.plot(continuous_true, color='lightgreen', linewidth=4, label='Traiettoria Reale')
        
        # Calcoliamo e disegniamo le predizioni
        with torch.no_grad():
            for i in range(num_windows):
                x_endo, x_exo, x_cond, _ = dataset[start_idx + i]
                
                inputs_e = x_endo.unsqueeze(0).to(device)
                inputs_ex = x_exo.unsqueeze(0).to(device)
                inputs_c = x_cond.unsqueeze(0).to(device)
                
                pred = model(inputs_e, exo=inputs_ex, cond=inputs_c).squeeze().cpu().numpy()
                
                # Asse X della predizione
                start_plot_x = lookback + (i * stride)
                x_axis_pred = range(start_plot_x, start_plot_x + horizon)
                
                # Disegniamo la linea di predizione con una leggera trasparenza
                label_pred = 'Predizione Rete (Horizon)' if i == 0 else ""
                plt.plot(x_axis_pred, pred, color='red', linewidth=1.5, alpha=0.6, label=label_pred)
                
                # IL NUOVO DETTAGLIO: Lo "Spillino" di ancoraggio all'inizio della predizione
                label_anchor = 'Punti di Ancoraggio' if i == 0 else ""
                plt.plot(start_plot_x, pred[0], marker='o', color='darkred', markersize=4, label=label_anchor)
                
        plt.axvline(x=lookback, color='gray', linestyle='--', linewidth=2)
        plt.ylabel("Estensione Braccio [0.0 - 1.0]", fontsize=12)
        plt.xlabel("Frame Temporali Continui (30Hz)", fontsize=12)
        plt.legend(loc='upper right', framealpha=0.9)
        plt.grid(True, alpha=0.3)
        plt.xlim(0, len(continuous_true) + horizon)
        plt.ylim(0, 1.05)
        plt.tight_layout()
        plt.show()
        
        # 4. AVANZAMENTO AL GIOCATORE SUCCESSIVO
        current_idx = end_idx
        player_count += 1
        
    print("\nScansione completata!")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def plot_all_players_full_session_advanced(model, dataset, stride=10, lookback=100, horizon=25, max_players=None):
    """
    Scansiona interamente il dataset in modalità BATCH (ultra-veloce) e genera 
    una dashboard a 3 pannelli per ogni giocatore:
    1. Cinematica (Reale vs Predetto + Dinamica di Gruppo)
    2. Props (Ostacoli e Bonus)
    3. Raycast (Distanza dalle sponde)
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.eval()
    
    current_idx = 0
    player_count = 0
    
    print(f"Inizio scansione avanzata del dataset (Totale campioni: {len(dataset)})...")
    
    while current_idx < len(dataset):
        if max_players is not None and player_count >= max_players:
            break
            
        print(f"\n--- Estrazione Giocatore {player_count + 1} ---")
        
        # ====================================================================
        # 1. IDENTIFICAZIONE SESSIONE E VARIABILI STATICHE (RowSide)
        # ====================================================================
        start_idx = current_idx
        first_e, first_ex, first_x_cond, _ = dataset[start_idx]
        
        # Identifichiamo il lato di remata (Indice 0 di x_cond è il Self_RowSide)
        rowside_val = first_x_cond[0].item()
        if rowside_val > 0.5:
            lato_remata = "Destra"
        elif rowside_val > -0.5:
            lato_remata = "Sinistra"
        else:
            lato_remata = "Sconosciuto (-1.0)"
        
        # Troviamo la fine della sessione
        end_idx = start_idx
        while end_idx < len(dataset):
            _, _, curr_x_cond, _ = dataset[end_idx]
            if not torch.allclose(curr_x_cond, first_x_cond):
                break
            end_idx += 1
            
        num_windows = end_idx - start_idx
        print(f"Trovate {num_windows} finestre. Avvio inferenza batch...")
        
        # ====================================================================
        # 2. RACCOLTA DATI E INFERENZA BATCH (Ultra-Veloce)
        # ====================================================================
        list_e, list_ex, list_c, list_y = [], [], [], []
        
        for i in range(num_windows):
            e, ex, c, y = dataset[start_idx + i]
            list_e.append(e)
            list_ex.append(ex)
            list_c.append(c)
            list_y.append(y)
            
        # Creiamo i mega-tensori per la GPU
        batch_e = torch.stack(list_e).to(device)
        batch_ex = torch.stack(list_ex).to(device)
        batch_c = torch.stack(list_c).to(device)
        
        # INFERENZA IN UN SOLO COLPO!
        with torch.no_grad():
            preds = model(batch_e, exo=batch_ex, cond=batch_c).squeeze(-1).cpu().numpy()
            
        # ====================================================================
        # 3. RICOSTRUZIONE DELLE SERIE TEMPORALI CONTINUE
        # ====================================================================
        # A) Cinematica Reale (Ground Truth del Self - Canale 0)
        continuous_true = list(first_e[0, :].numpy()) # Lookback iniziale
        for y in list_y:
            continuous_true.extend(y.squeeze().numpy()[:stride])
            
        # B) Contesto Ambientale (Preleviamo i dati da x_exo)
        # Indici: 6=Distractor_dz, 8=Attractor_dz, 12=RayLeft_dz, 14=RayRight_dz
        cont_dist_z = list(first_ex[6, :].numpy())
        cont_attr_z = list(first_ex[8, :].numpy())
        cont_ray_L  = list(first_ex[12, :].numpy())
        cont_ray_R  = list(first_ex[14, :].numpy())
        
        # C) Dinamica dei Compagni (Others - Canali da 1 a 7)
        cont_others = [list(first_e[c, :].numpy()) for c in range(1, 8)]
        
        # Aggiungiamo i nuovi 'stride' frame per ogni finestra successiva
        for idx_window, ex in enumerate(list_ex[1:]):
            cont_dist_z.extend(ex[6, -stride:].numpy())
            cont_attr_z.extend(ex[8, -stride:].numpy())
            cont_ray_L.extend(ex[12, -stride:].numpy())
            cont_ray_R.extend(ex[14, -stride:].numpy())
            
            # Estraiamo l'aggiornamento cinematico per i compagni dal tensore endogeno
            e_window = list_e[idx_window + 1]
            for c in range(1, 8):
                cont_others[c-1].extend(e_window[c, -stride:].numpy())
            
        x_axis_context = np.arange(len(cont_dist_z))

        # ====================================================================
        # 4. PLOTTING MULTI-PANNELLO
        # ====================================================================
        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(18, 14), gridspec_kw={'height_ratios': [2, 1, 1]})
        fig.suptitle(f"Analisi Completa - Giocatore {player_count + 1} | Lato Remata: {lato_remata}", fontsize=18, fontweight='bold')
        
        # --- PANNELLO 1: Cinematica e Predizioni ---
        ax1.set_title("Performance Modello: Traiettoria Reale vs Predizione (Horizon 75)", fontsize=14)
        ax1.axvspan(0, lookback, color='lightgray', alpha=0.3, label='Fase di Inizializzazione (Lookback)')
        
        # 1. Plot della Dinamica di Gruppo (Others in background)
        altri_attivi = False
        for c in range(7):
            # Se la media della linea è > -0.5, significa che c'è un giocatore reale e non un sedile vuoto
            if np.mean(cont_others[c]) > -0.5:
                altri_attivi = True
                ax1.plot(x_axis_context, cont_others[c], color='gray', alpha=0.35, linewidth=1.5)
        
        if altri_attivi:
            ax1.plot([], [], color='gray', alpha=0.35, linewidth=1.5, label='Compagni Attivi (Background)')

        # 2. Plot del Self (Ground Truth)
        ax1.plot(continuous_true, color='lightgreen', linewidth=4, label='Traiettoria Reale (Self)')
        
        # 3. Plot delle Predizioni della Rete
        for i in range(num_windows):
            start_plot_x = lookback + (i * stride)
            x_axis_pred = range(start_plot_x, start_plot_x + horizon)
            
            label_pred = 'Predizione Rete' if i == 0 else ""
            label_anchor = 'Ancoraggi' if i == 0 else ""
            
            ax1.plot(x_axis_pred, preds[i], color='red', linewidth=1.5, alpha=0.6, label=label_pred)
            ax1.plot(start_plot_x, preds[i][0], marker='o', color='darkred', markersize=4, label=label_anchor)
            
        ax1.axvline(x=lookback, color='gray', linestyle='--', linewidth=2)
        ax1.set_ylabel("Estensione Braccio", fontsize=12)
        ax1.legend(loc='upper right')
        ax1.grid(True, alpha=0.3)
        ax1.set_ylim(0, 1.05)
        ax1.set_xlim(0, len(continuous_true) + horizon)
        
        # Aggiunta badge informativo
        info_text = f"Input: {num_windows} batch\nStride: {stride}\nRowSide: {lato_remata}"
        ax1.text(0.01, 0.95, info_text, transform=ax1.transAxes, verticalalignment='top',
                 fontsize=11, fontweight='bold', color='#333333',
                 bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='gray'))

        # --- PANNELLO 2: Props (Ostacoli e Bonus) ---
        ax2.set_title("Distanza Longitudinale da Oggetti (Z)", fontsize=12, fontweight='bold')
        ax2.plot(x_axis_context, cont_dist_z, color='salmon', linewidth=2, label='Distrattori (Ostacoli)')
        ax2.plot(x_axis_context, cont_attr_z, color='gold', linewidth=2, label='Attrattori (Bonus)')
        ax2.axvline(x=lookback, color='gray', linestyle='--', linewidth=2)
        ax2.set_ylabel("Distanza Norm. [-1, 1]")
        ax2.legend(loc='upper right')
        ax2.grid(True, alpha=0.3)
        ax2.set_xlim(0, len(continuous_true) + horizon)
        ax2.set_ylim(-1.05, 1.05)

        # --- PANNELLO 3: Raycast (Sponde del Fiume) ---
        ax3.set_title("Sensori di Prossimità (Argini Destro e Sinistro)", fontsize=12, fontweight='bold')
        ax3.plot(x_axis_context, cont_ray_L, color='dodgerblue', linewidth=2, label='Sponda Sinistra (RayLeft Z)')
        ax3.plot(x_axis_context, cont_ray_R, color='purple', linewidth=2, label='Sponda Destra (RayRight Z)')
        ax3.axvline(x=lookback, color='gray', linestyle='--', linewidth=2)
        ax3.set_ylabel("Distanza Norm. [-1, 1]")
        ax3.set_xlabel("Frame Temporali Continui (30Hz)", fontsize=12)
        ax3.legend(loc='upper right')
        ax3.grid(True, alpha=0.3)
        ax3.set_xlim(0, len(continuous_true) + horizon)
        ax3.set_ylim(-1.05, 1.05)

        plt.tight_layout()
        plt.show()
        
        # ====================================================================
        # 5. AVANZAMENTO AL GIOCATORE SUCCESSIVO
        # ====================================================================
        current_idx = end_idx
        player_count += 1
        
    print("\nScansione completata!")

# ESEMPIO DI UTILIZZO:
# plot_all_players_full_session_advanced(model, test_dataset, stride=10, max_players=3)

In [ ]:
import os
import numpy as np
import torch
import pandas as pd
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

def analyze_frequencies(model, dataset, stride=10, lookback=100, horizon=25, fps=30, output_dir="storage"):
    """
    Scansiona il dataset, ricostruisce le serie continue e calcola la frequenza
    media (remate al secondo) per il Ground Truth e le Predizioni.
    Salva inoltre le serie temporali complete in file CSV all'interno della cartella specificata.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.eval()
    
    current_idx = 0
    player_count = 0
    
    results = []
    all_timeseries_data = [] # Lista per raccogliere i dataframe di tutte le serie temporali
    
    print(f"Inizio analisi frequenze sul dataset...")
    
    while current_idx < len(dataset):
        start_idx = current_idx
        _, _, first_x_cond, _ = dataset[start_idx]
        
        # Trova la fine della sessione del giocatore corrente
        end_idx = start_idx
        while end_idx < len(dataset):
            _, _, curr_x_cond, _ = dataset[end_idx]
            if not torch.allclose(curr_x_cond, first_x_cond):
                break
            end_idx += 1
            
        num_windows = end_idx - start_idx
        
        # 1. Costruiamo le serie continue
        continuous_true = []
        continuous_pred = []
        
        first_x_endo, _, _, _ = dataset[start_idx]
        # Inizializziamo il passato con il lookback
        continuous_true.extend(first_x_endo[0, :].numpy())
        continuous_pred.extend(first_x_endo[0, :].numpy()) # Per la predizione, il passato è noto
        
        with torch.no_grad():
            for i in range(num_windows):
                x_endo, x_exo, x_cond, y_target = dataset[start_idx + i]
                
                # Calcoliamo la predizione per questo stride
                inputs_e = x_endo.unsqueeze(0).to(device)
                inputs_ex = x_exo.unsqueeze(0).to(device)
                inputs_c = x_cond.unsqueeze(0).to(device)
                
                pred = model(inputs_e, exo=inputs_ex, cond=inputs_c).squeeze().cpu().numpy()
                
                # Allineamento perfetto: all'ultima finestra prendiamo tutto l'horizon,
                # altrimenti prendiamo solo lo stride.
                chunk_size = stride if i < num_windows - 1 else horizon
                
                # Aggiungiamo i dati reali e predetti sincronizzati
                nuovi_frame_reali = y_target.squeeze().numpy()[:chunk_size]
                
                continuous_true.extend(nuovi_frame_reali)
                continuous_pred.extend(pred[:chunk_size])
        
        # --- Salvataggio della Serie Temporale ---
        # Creiamo un DataFrame temporaneo per questo specifico giocatore/sessione
        df_ts = pd.DataFrame({
            'ID_Blocco_Sessione': player_count + 1,
            'Frame_Continuo': np.arange(len(continuous_true)),
            'Reale': continuous_true,
            'Predetto': continuous_pred
        })
        all_timeseries_data.append(df_ts)
        # ------------------------------------------------
        
        # 2. Calcolo dei Picchi e della Frequenza
        # Escludiamo il lookback dal calcolo delle frequenze per non "barare"
        y_true = np.array(continuous_true[lookback:]) 
        y_pred = np.array(continuous_pred[lookback:])
        
        # I parametri distance e prominence aiutano a non contare i falsi picchi
        # distance=20 significa almeno 2/3 di secondo tra una remata e l'altra a 30fps
        peaks_true, _ = find_peaks(y_true, prominence=0.15, distance=20)
        peaks_pred, _ = find_peaks(y_pred, prominence=0.15, distance=20)
        
        durata_sec = len(y_true) / fps
        
        # Frequenza in Hz (remate al secondo)
        freq_true = len(peaks_true) / durata_sec if durata_sec > 0 else 0
        freq_pred = len(peaks_pred) / durata_sec if durata_sec > 0 else 0
        
        # Calcolo errore
        error_hz = abs(freq_true - freq_pred)
        
        results.append({
            'ID_Blocco_Sessione': player_count + 1,
            'Durata_Analisi_Sec': round(durata_sec, 2),
            'Num_Remate_Reali': len(peaks_true),
            'Num_Remate_Predette': len(peaks_pred),
            'Freq_Reale_Hz': freq_true,
            'Freq_Predetta_Hz': freq_pred,
            'Errore_Hz': error_hz
        })
        
        current_idx = end_idx
        player_count += 1

    # Assicuriamoci che la cartella di output esista
    os.makedirs(output_dir, exist_ok=True)

    # 3. Creazione dei DataFrame Pandas e Salvataggio CSV nello storage
    df_results = pd.DataFrame(results)
    csv_riepilogo = os.path.join(output_dir, "Analisi_Frequenze_Riepilogo.csv")
    df_results.to_csv(csv_riepilogo, index=False)
    
    # Salvataggio del nuovo CSV Gigante con tutte le serie temporali nello storage
    df_all_ts = pd.concat(all_timeseries_data, ignore_index=True)
    csv_serie_temporali = os.path.join(output_dir, "Serie_Temporali_Reale_Vs_Predetto.csv")
    df_all_ts.to_csv(csv_serie_temporali, index=False)
    
    print(f"\nAnalisi completata!")
    print(f"1. Riepilogo frequenze salvato in: {csv_riepilogo}")
    print(f"2. Serie temporali complete salvate in: {csv_serie_temporali}")
    
    # Stampiamo un riepilogo
    print("\n=== RIEPILOGO FREQUENZE (Reale vs Predetto) ===")
    print(df_results[['ID_Blocco_Sessione', 'Freq_Reale_Hz', 'Freq_Predetta_Hz', 'Errore_Hz']].to_string(index=False))
    
    # 4. Piccolo plot riassuntivo
    plt.figure(figsize=(14, 6))
    x = np.arange(len(df_results))
    width = 0.35
    
    plt.bar(x - width/2, df_results['Freq_Reale_Hz'], width, label='Frequenza Reale (Hz)', color='lightgreen')
    plt.bar(x + width/2, df_results['Freq_Predetta_Hz'], width, label='Frequenza Predetta (Hz)', color='salmon')
    
    plt.ylabel('Frequenza (Remate al Secondo)')
    plt.xlabel('Sessione Giocatore (Indice Blocco)')
    plt.title('Confronto Frequenze di Remata: Ground Truth vs Rete Neurale')
    plt.xticks(x, df_results['ID_Blocco_Sessione'])
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

# --- ESEMPIO DI UTILIZZO ---
# analyze_frequencies(model, test_dataset, output_dir="../storage")

In [ ]:
lookback = 100
horizon = 25
batch_size = 2048
stride = 10 # stride is the step size for the sliding window when creating sequences from the time series data. It determines how much to move forward in the time series to create the next sequence. A smaller stride means more overlapping sequences, while a larger stride means less overlap and potentially fewer training samples.
# If you don't want to use stride, you can set it to 1, which means that the sliding window will move one step at a time, creating sequences with maximum overlap. This can be useful if you want to generate more training samples from your time series data, but it may also increase the training time and memory usage.

In [ ]:
from lib.Dataloaders.VirtualAgentDataset import VirtualAgentDataset

# # Percorso del tensore che abbiamo appena creato!
# DATA_PATH = "/home/jessica/storage/VA_Dataset_Tensor.npz"

# print("Caricamento dataset in memoria...")
# train_dataset = VirtualAgentDataset(DATA_PATH, split='train')
# val_dataset = VirtualAgentDataset(DATA_PATH, split='val')
# test_dataset = VirtualAgentDataset(DATA_PATH, split='test')

# train = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# val = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
# test = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

percorso_storage = os.path.abspath(os.path.join(os.getcwd(), "..", "storage"))

train_path = os.path.join(percorso_storage, "VA_Dataset_Train.npz")
val_path   = os.path.join(percorso_storage, "VA_Dataset_Val.npz")
test_path  = os.path.join(percorso_storage, "VA_Dataset_Test.npz")

train_dataset = VirtualAgentDataset(train_path, split='train')
val_dataset   = VirtualAgentDataset(val_path, split='val')
test_dataset  = VirtualAgentDataset(test_path, split='test')

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# model = SimpleTransformerForecast(lookback=lookback,
#                                   horizon=horizon,
#                                   input_dim=8,
#                                   target_dim=1,
#                                   d_model=16,
#                                   n_heads=2,
#                                   num_layers=6, dropout=0).to('cuda')
# model.compile()

In [ ]:
# model = PatchTransformerForecast(lookback=lookback,
#                                 horizon=horizon,
#                                 input_dim=8,
#                                 target_dim=1,
#                                 d_model=64,
#                                 n_heads=8,
#                                 num_layers=2, dropout=0, patch_len=10).to('cuda')

In [ ]:
# model = PatchTransformerForecast2(lookback=lookback,
#                                   horizon=horizon,
#                                   input_dim=8,
#                                   target_dim=1,
#                                   d_model=64,
#                                   n_heads=8,
#                                   num_layers=2,
#                                   norm_first=True,
#                                   dropout=0,
#                                   patch_len=10,
#                                   layer_norm_eps=1e-5,
#                                   bias=True,
#                                   swiglu=True,
#                                   rmsnorm=True,
#                                   trans_norm=True,
#                                   device='cuda').to('cuda')
# model.compile()

In [ ]:
# model = PatchTransformerExoForecast(
#     lookback=lookback,
#     horizon=horizon,
#     input_dim=8,           # 8 distanze passate (Self + 7 Others)
#     exo_dim=19,            # LE NOSTRE 19 FEATURE DELLA BARCA
#     condition_dim=16,       # RowSide + Score (se il tuo script usa cond_dim scrivilo pure)
#     target_dim=1,          # 1 distanza futura (Self)
#     d_model=64,
#     n_heads=8,
#     num_layers=2,
#     norm_first=True,
#     dropout=0,
#     patch_len=10,
#     layer_norm_eps=1e-5,
#     bias=True,
#     swiglu=True,
#     rmsnorm=True,
#     trans_norm=True,
#     device='cuda', 
#     verbose=False
# ).to('cuda')
# # model.compile()

In [ ]:
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
# scheduler = StepLR(optimizer, step_size=20, gamma=0.90)

In [ ]:
# ==============================================================================
# NUOVO MODELLO 
# ==============================================================================

model = PatchTransformerExoForecast(
    lookback=lookback,
    horizon=horizon,
    input_dim=8,           
    exo_dim=19,            
    condition_dim=16,       
    target_dim=1,          
    # --- INIZIO TUNING ---
    d_model=128,            # RIDOTTO da 64 a 32: Il modello era troppo "grande" per questo problema fisico e imparava a memoria.
    n_heads=8,              # RIDOTTO da 8 a 4: Meno teste di attenzione aiutano a generalizzare.
    num_layers=6,           # Lasciato a 2.
    dropout=0.3,            # AUMENTATO da 0.0 a 0.2. Spegne a caso il 20% dei neuroni costringendo la rete a capire la fisica vera.
    patch_len=5,            # (1/3 di secondo di patch)
    # --- FINE TUNING ---
    norm_first=True,
    layer_norm_eps=1e-5,
    bias=True,
    swiglu=True,
    rmsnorm=True,
    trans_norm=True,
    device='cuda', 
    verbose=False
).to('cuda')

In [ ]:
# ==============================================================================
# 3. NUOVO OPTIMIZER CON WEIGHT DECAY
# ==============================================================================

# Aggiungiamo 'weight_decay' (L2 Regularization). 
# È un'altra tecnica potentissima che "punisce" i pesi troppo grandi, riducendo l'overfitting.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Aumentiamo lo step_size: vogliamo che impari lentamente per più epoche
scheduler = StepLR(optimizer, step_size=50, gamma=0.90)

In [ ]:
# ==============================================================================
# 1. CACCIA ALL'ANOMALIA NEL VALIDATION SET
# Esegui questa cella per trovare quale campione sta facendo esplodere la Val Loss
# ==============================================================================
import torch.nn.functional as F
import numpy as np
import torch

def find_validation_outlier(dataset, model):
    print("Ricerca del campione anomalo in corso...")
    max_loss = 0.0
    outlier_idx = -1
    
    model.eval()
    with torch.no_grad():
        for i in range(len(dataset)):
            x_endo, x_exo, x_cond, y_target = dataset[i]
            
            # 1. Controllo manuale sui dati grezzi (Cerca picchi fisicamente impossibili)
            if torch.max(torch.abs(y_target)) > 2.0 or torch.max(torch.abs(x_exo)) > 5.0:
                print(f"!!! ATTENZIONE !!! Dati grezzi sballati all'indice {i}!")
                print(f"Max Y: {torch.max(torch.abs(y_target)).item():.2f}, Max Exo: {torch.max(torch.abs(x_exo)).item():.2f}")
            
            # 2. Controllo della Loss del Modello
            # Aggiungiamo la dimensione batch per darli in pasto al modello
            x_e = x_endo.unsqueeze(0).to('cuda')
            x_ex = x_exo.unsqueeze(0).to('cuda')
            x_c = x_cond.unsqueeze(0).to('cuda')
            y = y_target.unsqueeze(0).to('cuda')
            
            out = model(x_e, exo=x_ex, cond=x_c).squeeze(-1)
            loss = F.mse_loss(y.squeeze(1), out).item()
            
            if loss > max_loss:
                max_loss = loss
                outlier_idx = i
                
    print(f"\n--- RISULTATO ---")
    print(f"L'anomalia peggiore è all'indice: {outlier_idx} con una Loss di: {max_loss:.2f}")

# Lanciamo la caccia all'anomalia passandogli il dataset di validazione e il modello!
# NOTA: passiamo val_dataset (il Dataset puro), NON val (il DataLoader)
#find_validation_outlier(val_dataset, model)

In [ ]:
# # Testing the model with random input
# random_input = torch.randn(32, 8, 120).to('cuda')
# model.to('cuda')
# model(random_input).shape

In [ ]:
hist_loss = train_loop(model, train_loader, val_loader, optimizer, scheduler, epochs=1000, patience=100)

In [ ]:
plt.plot(hist_loss['train'], label='train')
plt.plot(hist_loss['val'], label='val')
plt.legend()


In [ ]:
print(scheduler.get_last_lr())

In [ ]:
#plot_prediction(model, train.dataset[10000]) 
plot_prediction(model, train_dataset[1000], titolo_extra="(Training Set)")
# TO DO: plottare tutta la serie temporale del soggetto self di una giocata intera non solo di un singolo frame ma di tutti i frame uno dopo l'altro.
plot_prediction(model, train_dataset[1001], titolo_extra="(Training Set)")

# to do: provare a fare in modo che il lookback abbiamo sempre INTERI cicli di movimento dentro la sua finestra così che due horizon consecutivi non abbiano un ciclo di movimento tagliato a metà. Questo potrebbe aiutare la rete a capire meglio la fisica del movimento e a fare previsioni più accurate.

# to do: si può predire l'horizon con sovrapposizione? 
# altrimenti interpolazione (in tempo reale) lineare o cubica 

In [ ]:
#plot_prediction(model, val.dataset[6])

plot_prediction(model, val_dataset[2822], titolo_extra="(Validation Set)")

In [ ]:
#plot_prediction(model, test.dataset[0])

plot_prediction(model, test_dataset[2822], titolo_extra="(Test Set)")
plot_prediction(model, test_dataset[2823], titolo_extra="(Test Set)")

In [ ]:
# # Proviamo la dashboard interattiva sul Test Set
# interactive_rolling_dashboard(model, test_dataset)

In [ ]:
plot_all_players_full_session(model, test_dataset, max_players=2)

In [ ]:
plot_all_players_full_session_advanced(model, test_dataset, stride=10, max_players=2)

In [ ]:
analyze_frequencies(model, test_dataset, output_dir="/home/jessica/TempSimul/storage")

In [ ]:
def dataset_loss(loader):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    data_loss = 0.0
    for i, (x_endo, x_exo, x_cond, y_target) in enumerate(loader):
        # Sposta su GPU
        x_endo = x_endo.to(device)
        x_exo = x_exo.to(device)
        x_cond = x_cond.to(device)
        y = y_target.to(device)
        
        # Predizione e calcolo errore
        outputs = model(x_endo, exo=x_exo, cond=x_cond).squeeze(-1)
        loss = F.mse_loss(y.squeeze(1), outputs)
        data_loss += loss.item()
        
    return data_loss / (i + 1)

print('Train loss:', dataset_loss(train_loader))
print('Val loss:', dataset_loss(val_loader))
print('Test loss:', dataset_loss(test_loader))

# dopo aver interpolato --> 
# frequenza media su serie reale horizon e su serie predetta horizon, vedere se la rete riesce a predire la frequenza media del movimento.

# chiedere al prof se il dateset che abbiamo ha valore per una pubblicazione scientifica, oppure se è solo un dataset di test per il progetto di tesi.

In [ ]:
# Dopo che l'addestramento è finito e sei soddisfatto
percorso_salvataggio = 'weights_transformer_patch3.pth'

# Salviamo SOLO lo state_dict (i pesi)
torch.save(model.state_dict(), percorso_salvataggio)
print(f"[*] Pesi salvati correttamente in: {percorso_salvataggio}")

In [ ]:
# se ti server fare il reload dei pesi in un secondo momento, puoi fare così:
import torch
# 1. Ricrea l'istanza del modello con gli STESSI IDENTICI parametri usati per il training
model = PatchTransformerExoForecast(
    lookback=lookback,
    horizon=horizon,
    input_dim=8,           
    exo_dim=19,            
    condition_dim=16,       
    target_dim=1,          
    # --- INIZIO TUNING ---
    d_model=128,            # RIDOTTO da 64 a 32: Il modello era troppo "grande" per questo problema fisico e imparava a memoria.
    n_heads=8,              # RIDOTTO da 8 a 4: Meno teste di attenzione aiutano a generalizzare.
    num_layers=6,           # Lasciato a 2.
    dropout=0.3,            # AUMENTATO da 0.0 a 0.2. Spegne a caso il 20% dei neuroni costringendo la rete a capire la fisica vera.
    patch_len=5,            # (1/3 di secondo di patch)
    # --- FINE TUNING ---
    norm_first=True,
    layer_norm_eps=1e-5,
    bias=True,
    swiglu=True,
    rmsnorm=True,
    trans_norm=True,
    device='cuda', 
    verbose=False
).to('cuda')

# 2. Carica i pesi salvati (sostituisci "model_weights.pth" con il nome reale del tuo file)
percorso_pesi = "/home/jessica/TempSimul/Notebooks/weights_transformer_patch.pth" 
model.load_state_dict(torch.load(percorso_pesi))

# 3. Metti il modello in modalità valutazione
model.eval()

# 4. Ricarica i Dataloader (o almeno il test_dataset che ti serve per i plot)
# (Assumendo che VirtualAgentDataset sia definito)
test_dataset = VirtualAgentDataset(npz_path="/home/jessica/TempSimul/storage/VA_Dataset_Test.npz", split='test')

# 5. Lancia la dashboard interattiva
#interactive_rolling_dashboard(model, test_dataset, stride=30, lookback=300, horizon=75)

In [ ]:
# %%
# 3. Generazione della tabella passandogli il modello e la forma esatta dell'input
from torchinfo import summary

# (Batch, Canali, Tempo)
summary(model, input_data=(
    torch.randn(1, 8, lookback).cuda(),      # x_endo (Batch, 8, Tempo)
),
    exo=torch.randn(1, 19, lookback).cuda(), # <--- MODIFICATO: 19 canali esogeni
    
    # Non ha più il tempo (lookback), è un token statico
    cond=torch.randn(1, 16).cuda(),          # <--- MODIFICATO: 16 canali statici
    
    device='cuda',
    depth=4,
    mode="eval"
)

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

def export_testset_to_csv(model, dataset, output_path, stride=10, lookback=100, horizon=25):
    """
    Scansiona il Test Set in batch, esegue il Rolling Forecast continuo 
    (tenendo 'stride' frame e scartando la coda dell''horizon') e salva 
    TUTTE le variabili contestuali allineate temporalmente in un CSV per MATLAB.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.eval()
    
    current_idx = 0
    player_count = 0
    
    all_blocks_data = [] # Qui accumuleremo i DataFrame di tutti i giocatori
    
    print(f"Inizio estrazione dati per MATLAB (Campioni totali: {len(dataset)})...")
    
    while current_idx < len(dataset):
        start_idx = current_idx
        first_e, first_ex, first_x_cond, _ = dataset[start_idx]
        
        # Identifichiamo il lato di remata (RowSide del Self)
        rowside_val = first_x_cond[0].item()
        
        # Troviamo quante finestre ha questo giocatore
        end_idx = start_idx
        while end_idx < len(dataset):
            _, _, curr_x_cond, _ = dataset[end_idx]
            if not torch.allclose(curr_x_cond, first_x_cond):
                break
            end_idx += 1
            
        num_windows = end_idx - start_idx
        
        # --- 1. INFERENZA BATCH ULTRA-VELOCE ---
        list_e, list_ex, list_c, list_y = [], [], [], []
        for i in range(num_windows):
            e, ex, c, y = dataset[start_idx + i]
            list_e.append(e)
            list_ex.append(ex)
            list_c.append(c)
            list_y.append(y)
            
        batch_e = torch.stack(list_e).to(device)
        batch_ex = torch.stack(list_ex).to(device)
        batch_c = torch.stack(list_c).to(device)
        
        with torch.no_grad():
            preds = model(batch_e, exo=batch_ex, cond=batch_c).squeeze(-1).cpu().numpy()
            
        # --- 2. RICOSTRUZIONE TEMPORALE CONTINUA ---
        # Inizializziamo i dizionari per raccogliere i dati col Lookback iniziale
        # Per la predizione mettiamo NaN nel lookback (non c'è predizione nel passato)
        data_dict = {
            'ID_Blocco_Sessione': [player_count + 1] * lookback,
            'Frame': list(range(lookback)),
            'Self_Reale': list(first_e[0, :].numpy()),
            'Self_Predetto': [np.nan] * lookback,
            'RowSide': [rowside_val] * lookback,
            
            # Esogeni (Indici specifici estratti dal payload)
            'Boat_Angle': list(first_ex[2, :].numpy()),
            'Metronomo': list(first_ex[3, :].numpy()),
            'TeamScore': list(first_ex[4, :].numpy()),
            'Distractor_X': list(first_ex[5, :].numpy()),
            'Distractor_Z': list(first_ex[6, :].numpy()),
            'Attractor_X': list(first_ex[7, :].numpy()),
            'Attractor_Z': list(first_ex[8, :].numpy()),
            'RayFwd_X': list(first_ex[9, :].numpy()),
            'RayFwd_Z': list(first_ex[10, :].numpy()),
            'RayLeft_X': list(first_ex[11, :].numpy()),
            'RayLeft_Z': list(first_ex[12, :].numpy()),
            'RayRight_X': list(first_ex[13, :].numpy()),
            'RayRight_Z': list(first_ex[14, :].numpy()),
            'Ray45_X': list(first_ex[15, :].numpy()),
            'Ray45_Z': list(first_ex[16, :].numpy()),
            'Ray135_X': list(first_ex[17, :].numpy()),
            'Ray135_Z': list(first_ex[18, :].numpy())
        }
        
        # Aggiungiamo dinamicamente i 7 compagni di barca
        for c in range(1, 8):
            data_dict[f'Other_{c}'] = list(first_e[c, :].numpy())

        # --- 3. AGGANCIO DEGLI STRIDE (Il Futuro) ---
        current_frame = lookback
        for i in range(num_windows):
            # All'ultimo step prendiamo tutto l'horizon residuo, altrimenti solo lo stride
            chunk_size = stride if i < num_windows - 1 else horizon
            
            # Target Reale e Predetto (Dall'Horizon)
            y_real_chunk = list_y[i].squeeze().numpy()[:chunk_size]
            y_pred_chunk = preds[i][:chunk_size]
            
            # --- Recupero Contesto Ambientale ---
            # Il contesto per questi frame futuri si trova negli ULTIMI frame 
            # del Lookback della finestra SUCCESSIVA.
            if i < num_windows - 1:
                next_e = list_e[i + 1]
                next_ex = list_ex[i + 1]
                
                # Aggiungiamo i dati di contesto
                data_dict['Boat_Angle'].extend(next_ex[2, -chunk_size:].numpy())
                data_dict['Metronomo'].extend(next_ex[3, -chunk_size:].numpy())
                data_dict['TeamScore'].extend(next_ex[4, -chunk_size:].numpy())
                data_dict['Distractor_X'].extend(next_ex[5, -chunk_size:].numpy())
                data_dict['Distractor_Z'].extend(next_ex[6, -chunk_size:].numpy())
                data_dict['Attractor_X'].extend(next_ex[7, -chunk_size:].numpy())
                data_dict['Attractor_Z'].extend(next_ex[8, -chunk_size:].numpy())
                
                # Raycast
                data_dict['RayFwd_X'].extend(next_ex[9, -chunk_size:].numpy())
                data_dict['RayFwd_Z'].extend(next_ex[10, -chunk_size:].numpy())
                data_dict['RayLeft_X'].extend(next_ex[11, -chunk_size:].numpy())
                data_dict['RayLeft_Z'].extend(next_ex[12, -chunk_size:].numpy())
                data_dict['RayRight_X'].extend(next_ex[13, -chunk_size:].numpy())
                data_dict['RayRight_Z'].extend(next_ex[14, -chunk_size:].numpy())
                data_dict['Ray45_X'].extend(next_ex[15, -chunk_size:].numpy())
                data_dict['Ray45_Z'].extend(next_ex[16, -chunk_size:].numpy())
                data_dict['Ray135_X'].extend(next_ex[17, -chunk_size:].numpy())
                data_dict['Ray135_Z'].extend(next_ex[18, -chunk_size:].numpy())
                
                for c in range(1, 8):
                    data_dict[f'Other_{c}'].extend(next_e[c, -chunk_size:].numpy())
                    
            else:
                # Per gli ultimissimi frame (la coda dell'horizon), non abbiamo 
                # una finestra successiva da cui rubare il contesto. Pad con NaN per MATLAB.
                pad_nan = [np.nan] * chunk_size
                data_dict['Boat_Angle'].extend(pad_nan)
                data_dict['Metronomo'].extend(pad_nan)
                data_dict['TeamScore'].extend(pad_nan)
                data_dict['Distractor_X'].extend(pad_nan)
                data_dict['Distractor_Z'].extend(pad_nan)
                data_dict['Attractor_X'].extend(pad_nan)
                data_dict['Attractor_Z'].extend(pad_nan)
                data_dict['RayFwd_X'].extend(pad_nan)
                data_dict['RayFwd_Z'].extend(pad_nan)
                data_dict['RayLeft_X'].extend(pad_nan)
                data_dict['RayLeft_Z'].extend(pad_nan)
                data_dict['RayRight_X'].extend(pad_nan)
                data_dict['RayRight_Z'].extend(pad_nan)
                data_dict['Ray45_X'].extend(pad_nan)
                data_dict['Ray45_Z'].extend(pad_nan)
                data_dict['Ray135_X'].extend(pad_nan)
                data_dict['Ray135_Z'].extend(pad_nan)
                for c in range(1, 8):
                    data_dict[f'Other_{c}'].extend(pad_nan)

            # Aggiungiamo Reale e Predetto
            data_dict['Self_Reale'].extend(y_real_chunk)
            data_dict['Self_Predetto'].extend(y_pred_chunk)
            data_dict['RowSide'].extend([rowside_val] * chunk_size)
            data_dict['ID_Blocco_Sessione'].extend([player_count + 1] * chunk_size)
            data_dict['Frame'].extend(list(range(current_frame, current_frame + chunk_size)))
            
            current_frame += chunk_size

        # Convertiamo il dizionario di questo giocatore in un DataFrame e lo salviamo
        df_player = pd.DataFrame(data_dict)
        all_blocks_data.append(df_player)
        
        current_idx = end_idx
        player_count += 1
        print(f"  -> Estratto Blocco {player_count} (Frame totali: {len(df_player)})")

    # --- 4. SALVATAGGIO FINALE GIGANTE ---
    print("\nConcatenazione di tutti i blocchi in corso...")
    df_final = pd.concat(all_blocks_data, ignore_index=True)
    
    # Arrotondiamo a 4 decimali per alleggerire il file CSV
    df_final = df_final.round(4)
    
    df_final.to_csv(output_path, index=False)
    print(f"VITTORIA! CSV generato con successo in: {output_path}")
    print(f"Dimensioni totali: {df_final.shape[0]} righe x {df_final.shape[1]} colonne.")

# --- ESEMPIO DI UTILIZZO ---
percorso_export = os.path.join("/home/jessica/TempSimul/storage", "Esportazione_MATLAB_Completa.csv")
export_testset_to_csv(model, test_dataset, output_path=percorso_export, stride=10, lookback=100, horizon=25)